# Preprocess ArtEmis: Link to WikiArt Image Paths

This notebook links `painting` and `anchor_painting` from `Contrastive.csv` to relative image paths inside the WikiArt dataset.

What this notebook does:
1. Loads `datasets/ArtEmis/Contrastive.csv`
2. Scans all image files under `datasets/Wikiart/`
3. Builds a mapping from image name (filename without extension) to relative path
4. Adds `painting_path` and `anchor_painting_path` columns
5. Saves `datasets/ArtEmis/Contrastive_with_paths.csv`

In [7]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    """Walk up from the current directory until the expected dataset layout is found."""
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        has_artemis = (candidate / "datasets" / "ArtEmis" / "Contrastive.csv").exists()
        has_wikiart = (candidate / "datasets" / "Wikiart").exists()
        if has_artemis and has_wikiart:
            return candidate
    raise FileNotFoundError(
        "Could not locate project root containing datasets/ArtEmis/Contrastive.csv and datasets/Wikiart"
    )

# Resolve project root robustly, even if the notebook is run from datasets/ArtEmis.
project_root = find_project_root(Path.cwd())

contrastive_csv = project_root / "datasets" / "ArtEmis" / "Contrastive.csv"
wikiart_root = project_root / "datasets" / "Wikiart"
output_csv = project_root / "datasets" / "ArtEmis" / "Contrastive_with_paths.csv"

print("Project root:", project_root)
print("Input CSV:", contrastive_csv)
print("WikiArt root:", wikiart_root)
print("Output CSV:", output_csv)

if not contrastive_csv.exists():
    raise FileNotFoundError(f"Could not find input CSV: {contrastive_csv}")
if not wikiart_root.exists():
    raise FileNotFoundError(f"Could not find WikiArt folder: {wikiart_root}")

Project root: c:\Users\Thijs\Desktop\Ai Art Critic
Input CSV: c:\Users\Thijs\Desktop\Ai Art Critic\datasets\ArtEmis\Contrastive.csv
WikiArt root: c:\Users\Thijs\Desktop\Ai Art Critic\datasets\Wikiart
Output CSV: c:\Users\Thijs\Desktop\Ai Art Critic\datasets\ArtEmis\Contrastive_with_paths.csv


In [8]:
df = pd.read_csv(contrastive_csv)

print("Loaded rows:", len(df))
print("Columns:", list(df.columns))
display(df.head(3))

Loaded rows: 237998
Columns: ['emotion', 'utterance', 'art_style', 'painting', 'anchor_art_style', 'anchor_painting', 'repetition']


,emotion,utterance,art_style,painting,anchor_art_style,anchor_painting,repetition
0,sadness,This painting shows a gloomy day. The darkness...,Realism,johan-hendrik-weissenbruch_windy-day,Realism,johan-hendrik-weissenbruch_ship-canal,1
1,sadness,the weather and atmosphere makes me feel sad,Romanticism,ivan-aivazovsky_landscape-with-a-schooner-1880,Realism,johan-hendrik-weissenbruch_ship-canal,45
2,sadness,"Dark and moody, cloudy sky that is depressing",Impressionism,paul-gauguin_clearing-1873,Realism,johan-hendrik-weissenbruch_ship-canal,18


In [9]:
# File types to include when scanning WikiArt images.
valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

image_files = [
    p for p in wikiart_root.rglob("*")
    if p.is_file() and p.suffix.lower() in valid_ext
]

print(f"Found {len(image_files):,} image files inside WikiArt.")

# Map: filename stem -> list of relative paths (relative to datasets folder).
name_to_paths = {}
for img_path in image_files:
    stem = img_path.stem.strip().lower()
    rel_to_datasets = img_path.relative_to(project_root / "datasets")
    name_to_paths.setdefault(stem, []).append(rel_to_datasets.as_posix())

print(f"Unique filename stems mapped: {len(name_to_paths):,}")

# Show a few example mappings for transparency.
sample_items = list(name_to_paths.items())[:5]
for stem, paths in sample_items:
    print(f"- {stem} -> {paths[0]}")

Found 81,444 image files inside WikiArt.
Unique filename stems mapped: 80,095
- aaron-siskind_acolman-1-1955 -> Wikiart/Abstract_Expressionism/aaron-siskind_acolman-1-1955.jpg
- aaron-siskind_chicago-1951 -> Wikiart/Abstract_Expressionism/aaron-siskind_chicago-1951.jpg
- aaron-siskind_chicago-6-1961 -> Wikiart/Abstract_Expressionism/aaron-siskind_chicago-6-1961.jpg
- aaron-siskind_feet-102-1957 -> Wikiart/Abstract_Expressionism/aaron-siskind_feet-102-1957.jpg
- aaron-siskind_gloucester-16a-1944 -> Wikiart/Abstract_Expressionism/aaron-siskind_gloucester-16a-1944.jpg


In [10]:
def resolve_image_path(name: str):
    if pd.isna(name):
        return None

    key = str(name).strip().lower()
    matches = name_to_paths.get(key, [])

    if not matches:
        return None

    # If duplicates exist, keep first path for deterministic output.
    return sorted(matches)[0]

df["painting_path"] = df["painting"].apply(resolve_image_path)
df["anchor_painting_path"] = df["anchor_painting"].apply(resolve_image_path)

missing_painting = df["painting_path"].isna().sum()
missing_anchor = df["anchor_painting_path"].isna().sum()

print("Missing painting_path values:", int(missing_painting))
print("Missing anchor_painting_path values:", int(missing_anchor))

display(df[["painting", "painting_path", "anchor_painting", "anchor_painting_path"]].head(10))

# Keep only valid rows where BOTH paths exist.
rows_before_filter = len(df)
df = df.dropna(subset=["painting_path", "anchor_painting_path"]).copy()
rows_after_filter = len(df)
rows_removed = rows_before_filter - rows_after_filter

print("Rows before filtering:", rows_before_filter)
print("Rows removed (missing either path):", rows_removed)
print("Rows remaining for export:", rows_after_filter)

Missing painting_path values: 2577
Missing anchor_painting_path values: 2194


,painting,painting_path,anchor_painting,anchor_painting_path
0,johan-hendrik-weissenbruch_windy-day,Wikiart/Realism/johan-hendrik-weissenbruch_win...,johan-hendrik-weissenbruch_ship-canal,Wikiart/Realism/johan-hendrik-weissenbruch_shi...
1,ivan-aivazovsky_landscape-with-a-schooner-1880,Wikiart/Romanticism/ivan-aivazovsky_landscape-...,johan-hendrik-weissenbruch_ship-canal,Wikiart/Realism/johan-hendrik-weissenbruch_shi...
2,paul-gauguin_clearing-1873,Wikiart/Impressionism/paul-gauguin_clearing-18...,johan-hendrik-weissenbruch_ship-canal,Wikiart/Realism/johan-hendrik-weissenbruch_shi...
3,john-henry-twachtman_winter-landscape,Wikiart/Impressionism/john-henry-twachtman_win...,johan-hendrik-weissenbruch_ships-in-harbour,Wikiart/Realism/johan-hendrik-weissenbruch_shi...
4,eugene-boudin_trouville-the-port-1864,Wikiart/Impressionism/eugene-boudin_trouville-...,johan-hendrik-weissenbruch_ships-in-harbour,Wikiart/Realism/johan-hendrik-weissenbruch_shi...
5,eugene-boudin_trouville-the-port-1864,Wikiart/Impressionism/eugene-boudin_trouville-...,johan-hendrik-weissenbruch_ships-in-harbour,Wikiart/Realism/johan-hendrik-weissenbruch_shi...
6,eugene-boudin_trouville-the-port-1864,Wikiart/Impressionism/eugene-boudin_trouville-...,johan-hendrik-weissenbruch_ships-in-harbour,Wikiart/Realism/johan-hendrik-weissenbruch_shi...
7,konstantin-makovsky_landscape-2,Wikiart/Realism/konstantin-makovsky_landscape-...,johan-hendrik-weissenbruch_summer-day,Wikiart/Realism/johan-hendrik-weissenbruch_sum...
8,ivan-aivazovsky_windmill-on-the-sea-coast-1851,Wikiart/Romanticism/ivan-aivazovsky_windmill-o...,johan-hendrik-weissenbruch_view-at-geestbrug,Wikiart/Realism/johan-hendrik-weissenbruch_vie...
9,johan-hendrik-weissenbruch_ships-on-canal,Wikiart/Realism/johan-hendrik-weissenbruch_shi...,johan-hendrik-weissenbruch_view-at-geestbrug,Wikiart/Realism/johan-hendrik-weissenbruch_vie...


Rows before filtering: 237998
Rows removed (missing either path): 4523
Rows remaining for export: 233475


In [11]:
df.to_csv(output_csv, index=False)
print(f"Saved new file: {output_csv}")
print(f"Exported {len(df):,} rows after removing rows with missing painting or anchor paths.")
print("Done. Original columns were preserved, and new path columns were added.")

Saved new file: c:\Users\Thijs\Desktop\Ai Art Critic\datasets\ArtEmis\Contrastive_with_paths.csv
Exported 233,475 rows after removing rows with missing painting or anchor paths.
Done. Original columns were preserved, and new path columns were added.
